# 常规失真与几何变换攻击闭环运行入口

该 Notebook 用于 Colab 或本地 CPU 会话: 挂载 Google Drive、拉取仓库、读取前序 aligned rescoring 与 threshold calibration 包, 对真实 source image 执行常规失真和几何变换攻击, 写出 attacked image、source / attacked digest、正式检测记录和总体进度, 并把核对文件打包到 Google Drive。

正式逻辑位于 `paper_workflow/colab_utils/conventional_geometric_attack_evaluation.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 本入口不需要 GPU runtime, 但也可以在 GPU 会话中运行。
2. 确认 Google Drive 中已有 aligned rescoring 包, 默认查找 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `aligned_rescoring/aligned_rescoring_package_*.zip``。
3. 确认 Google Drive 中已有 threshold calibration 包, 默认查找 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `threshold_calibration/threshold_calibration_package_*.zip``。
4. 默认输出目录为 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `conventional_geometric_attack_evaluation``。
5. 该入口只覆盖 `standard_distortion` 与 `geometric_transform`; `regeneration_attack` 仍由 `real_attack_evaluation_run.ipynb` 覆盖。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
SLM_WM_PAPER_RUN_NAME = "pilot_paper"
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
# 避免在已启动的 Colab 内核中升级 numpy, 否则可能造成 numpy 扩展与 Python 层不一致。
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub pillow tqdm


In [ ]:
import importlib.metadata as importlib_metadata


def package_version_or_missing(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return 'not_installed'


dependency_report = {
    'pillow': package_version_or_missing('pillow'),
    'numpy': package_version_or_missing('numpy'),
    'tqdm': package_version_or_missing('tqdm'),
}
print(dependency_report)


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="conventional_geometric_attack_evaluation",
)
print(paper_run_environment)


In [ ]:
import os

from paper_workflow.colab_utils.conventional_geometric_attack_evaluation import (
    run_default_conventional_geometric_attack_evaluation_from_drive_plan,
)

conventional_geometric_attack_summary = run_default_conventional_geometric_attack_evaluation_from_drive_plan(
    root='.',
    aligned_rescoring_drive_dir=os.environ['SLM_WM_ALIGNED_RESCORING_DRIVE_DIR'],
    threshold_calibration_drive_dir=os.environ['SLM_WM_THRESHOLD_CALIBRATION_DRIVE_DIR'],
    require_threshold_package=True,
)
assert conventional_geometric_attack_summary['run_decision'] == 'pass', conventional_geometric_attack_summary
assert conventional_geometric_attack_summary['real_attacked_image_closed_loop_ready'] is True, conventional_geometric_attack_summary
assert conventional_geometric_attack_summary['formal_attack_detection_ready'] is True, conventional_geometric_attack_summary
conventional_geometric_attack_summary


In [ ]:
from pathlib import Path

summary_path = Path('outputs/conventional_geometric_attack_evaluation/conventional_geometric_attack_run_summary.json')
input_manifest_path = Path('outputs/conventional_geometric_attack_evaluation/conventional_geometric_attack_input_package_manifest.json')
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else conventional_geometric_attack_summary)
if input_manifest_path.exists():
    print(input_manifest_path.read_text(encoding='utf-8'))


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.conventional_geometric_attack_evaluation import (
    package_conventional_geometric_attack_evaluation_outputs,
)


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ['SLM_WM_CONVENTIONAL_GEOMETRIC_ATTACK_DRIVE_DIR']
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%S%fZ').lower()
archive_name = f"conventional_geometric_attack_evaluation_package_{utc_suffix}_{resolve_short_commit()}.zip"
archive_record = package_conventional_geometric_attack_evaluation_outputs(
    root='.',
    drive_output_dir=drive_output_dir,
    archive_name=archive_name,
)
archive_record.to_dict()


In [ ]:
from pathlib import Path

output_root = Path('outputs/conventional_geometric_attack_evaluation')
for result_path in sorted(output_root.glob('*.json')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

for result_path in sorted(output_root.glob('*.csv')):
    if result_path.is_file():
        print(result_path)
        print(result_path.read_text(encoding='utf-8')[:4000])

records_path_text = conventional_geometric_attack_summary.get('output_records_path') or ''
formal_records_path_text = conventional_geometric_attack_summary.get('formal_records_path') or ''
for record_path_text in (records_path_text, formal_records_path_text):
    if record_path_text:
        path = Path(record_path_text)
        if path.exists():
            lines = path.read_text(encoding='utf-8').splitlines()
            print({'path': str(path), 'line_count': len(lines), 'preview': lines[:3]})
